# Joshi Part 7: Path-Dependent Options (Python)

Asian options, barrier options, and lookback options via Monte Carlo.

In [ ]:
import math
import numpy as np

## 1. Arithmetic Asian Call

In [ ]:
def asian_call_mc(S, K, r, sigma, T, num_dates, num_paths, seed=42):
    rng = np.random.default_rng(seed)
    dt = T / num_dates
    drift = (r - 0.5 * sigma**2) * dt
    vol_sqrt_dt = sigma * math.sqrt(dt)
    discount = math.exp(-r * T)
    
    payoffs = np.zeros(num_paths)
    for p in range(num_paths):
        spot = S
        total = 0.0
        for _ in range(num_dates):
            spot *= math.exp(drift + vol_sqrt_dt * rng.standard_normal())
            total += spot
        avg = total / num_dates
        payoffs[p] = max(avg - K, 0)
    
    price = discount * payoffs.mean()
    se = discount * payoffs.std(ddof=1) / math.sqrt(num_paths)
    return price, se

price, se = asian_call_mc(100, 100, 0.05, 0.2, 1.0, 12, 100_000)
print(f"Arithmetic Asian Call: {price:.4f} ± {se:.4f}")
print(f"(Should be < European Call ≈ 10.45)")

## 2. Up-and-Out Barrier Call

In [ ]:
def barrier_up_out_call_mc(S, K, barrier, r, sigma, T, num_dates, num_paths, seed=42):
    rng = np.random.default_rng(seed)
    dt = T / num_dates
    drift = (r - 0.5 * sigma**2) * dt
    vol_sqrt_dt = sigma * math.sqrt(dt)
    discount = math.exp(-r * T)
    
    payoffs = np.zeros(num_paths)
    for p in range(num_paths):
        spot = S
        knocked_out = False
        for _ in range(num_dates):
            spot *= math.exp(drift + vol_sqrt_dt * rng.standard_normal())
            if spot >= barrier:
                knocked_out = True
                break
        payoffs[p] = 0.0 if knocked_out else max(spot - K, 0)
    
    price = discount * payoffs.mean()
    se = discount * payoffs.std(ddof=1) / math.sqrt(num_paths)
    return price, se

for barrier in [120, 130, 150, 200]:
    price, se = barrier_up_out_call_mc(100, 100, barrier, 0.05, 0.2, 1.0, 252, 100_000)
    print(f"Barrier={barrier}: {price:.4f} ± {se:.4f}")

print(f"\nVanilla Call ≈ 10.45 (barrier → ∞)")

## 3. Lookback Call: payoff = S(T) - min S(t)

In [ ]:
def lookback_call_mc(S, r, sigma, T, num_dates, num_paths, seed=42):
    rng = np.random.default_rng(seed)
    dt = T / num_dates
    drift = (r - 0.5 * sigma**2) * dt
    vol_sqrt_dt = sigma * math.sqrt(dt)
    discount = math.exp(-r * T)
    
    payoffs = np.zeros(num_paths)
    for p in range(num_paths):
        spot = S
        min_spot = S
        for _ in range(num_dates):
            spot *= math.exp(drift + vol_sqrt_dt * rng.standard_normal())
            min_spot = min(min_spot, spot)
        payoffs[p] = spot - min_spot
    
    price = discount * payoffs.mean()
    se = discount * payoffs.std(ddof=1) / math.sqrt(num_paths)
    return price, se

price, se = lookback_call_mc(100, 0.05, 0.2, 1.0, 252, 100_000)
print(f"Lookback Call: {price:.4f} ± {se:.4f}")
print(f"(Should be > Vanilla Call ≈ 10.45)")

## Exotic Option Price Comparison

| Option | Price | Relation to Vanilla |
|--------|-------|---------|
| Vanilla Call | ≈ 10.45 | Baseline |
| Asian Call | < 10.45 | Averaging reduces volatility |
| Barrier (up-and-out) | < 10.45 | Knock-out reduces value |
| Lookback | > 10.45 | Optimal strike adds value |